## Non reversible markov chain

We study here the implementation of the algorithm for the real interesting case, i.e, for non reversible chains.

The considered chain is represented by the following $4 \times 4$ Markov kernel: 

$P = \begin{pmatrix}0.5 & 0.4 & 0 & 0.1 \\ 0.2 & 0.3 & 0.5 & 0 \\ 0 & 0.1 & 0.6 & 0. 3 \\ 0.2 & 0 & 0.3 & 0.5 \end{pmatrix}$

In [14]:
import numpy as np
dim = 4

# Basis

eye = np.identity(dim)
basis = np.asarray([row.reshape(-1,1) for row in eye])

# Markov kernel P

P = np.array([[0.5, 0.4, 0, 0.1], [0.2, 0.3, 0.5, 0], 
[0, 0.1, 0.6, 0.3], [0.2, 0, 0.3, 0.5]])

eigvals, eigvecs = np.linalg.eig(P.T)

# Finding the index of the eigenvalue equal to 1

index = np.where(np.isclose(eigvals, 1))[0][0]

# Retreiving the stationary distribution and normalizing it

pi = np.real(eigvecs[:, index])
pi = pi / sum(pi)

# defining pi state

pi_state = np.sqrt(pi).reshape(-1, 1)

print(f"Markov kernel loaded: \n{P}")

print(f"Stationary distribution: {pi}")
print(f"State of the stationary distribution: {pi_state}")

# Non reversability check

reversible = True

for i in range(P.shape[0]):
    for j in range(P.shape[0]):

        if not np.allclose(pi[i]*P[i, j],  pi[j]*P[j, i]):
            reversible = False
            break
    
    if not reversible:
        break

assert not reversible, print("Chain is reversible")
print("The chain is not reversible")

# Computing the time reversal chain

P_rev = np.zeros((4, 4))

for i in range(P.shape[0]):
    for j in range(P.shape[0]):

        P_rev[i][j] = (pi[j]/pi[i]) * P[j][i]

print(f"Time reversal chain: \n{P_rev}")

Markov kernel loaded: 
[[0.5 0.4 0.  0.1]
 [0.2 0.3 0.5 0. ]
 [0.  0.1 0.6 0.3]
 [0.2 0.  0.3 0.5]]
Stationary distribution: [0.17161716 0.15511551 0.39933993 0.27392739]
State of the stationary distribution: [[0.41426702]
 [0.39384707]
 [0.63193349]
 [0.52338073]]
The chain is not reversible
Time reversal chain: 
[[0.5        0.18076923 0.         0.31923077]
 [0.44255319 0.3        0.25744681 0.        ]
 [0.         0.19421488 0.6        0.20578512]
 [0.0626506  0.         0.4373494  0.5       ]]


## Main operators

The goal is to implement the following circuit:

$\\ (\bra{01} \otimes \square_P^\dagger)G(\ket{01} \otimes \square_P) \\$
To do so, the first step is to define the following operators:
$\\
    \square_P = \sum_{x \in S}\ket{x}\ket{P(x, .)}\bra{x} \\
    \square_{P^*} = \sum_{x \in S}\ket{x}\ket{P^*(x, .)}\bra{x} \\
    S = \sum_{x, y \in S} \ket{x, y}\bra{y, x}
$

And $G$ is obtained via GQSP. This time the chebyschev polynomial is applied to a different walk operator which is:

$\\ W = (2\widehat{\square}\widehat{\square}^\dagger - I)\widehat{S} \\$

where

$\\ \widehat{\square} = (\ket{0}\bra{0} \otimes \square_{P^*} + \ket{1}\bra{1} \otimes \square_P) \\ 
\widehat{S} = (\ket{0}\bra{0} \otimes S + \ket{1}\bra{1} \otimes S^\dagger)(X \otimes I)$


In [15]:
# Defining the isometries

square = np.zeros((dim * dim, dim))
square_rev = np.zeros((dim * dim, dim)) # both have dim = 16 x 4

for x in range(dim):
    for y in range(dim):
        square += np.sqrt(P[x][y]) * np.kron(basis[x], basis[y]) @ basis[x].conj().T
        square_rev += np.sqrt(P_rev[x][y]) * np.kron(basis[x], basis[y]) @ basis[x].conj().T

# Defining the swap operator

swap = np.zeros(((dim * dim), (dim * dim))) # dim 16 x 16

for x in range(dim):
    for y in range(dim):
        swap += np.kron(basis[x], basis[y]) @ np.kron(basis[y].conj().T, basis[x].conj().T)

# Defining hermitized square

proj_0 = np.array([[1, 0], [0, 0]], dtype=complex)
proj_1 = np.array([[0, 0], [0, 1]], dtype=complex) # both have dim 2 x 2

square_herm = np.kron(proj_0, square_rev) + np.kron(proj_1, square) # dim 32 x 8

# Defining hermitized swap

X_gate = np.array([[0, 1], [1, 0]], dtype=complex) # dim 2 x 2

I_space = np.eye(dim * dim, dtype=complex) # dim 16 x 16

swap_herm = (np.kron(proj_0, swap) + np.kron(proj_1, swap))@(np.kron(X_gate, I_space)) # dim 32 x 32

# Defining the hermitized walk operator

I_space_large = np.eye(dim*dim*2) # 32 x 32

W_herm = (2*square_herm@square_herm.conj().T - I_space_large)@swap_herm # dim 32 x 32

print(f"Walk operator: {W_herm}")

# Sanity checks

assert np.allclose(W_herm @ W_herm.conj().T, np.eye(dim * dim * 2)), print("Critical error, walk operator is not unitary")

print("Walk operator is unitary")

print(f"Number of non zero elements: {np.count_nonzero(W_herm)}")


Walk operator: [[0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 ...
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]]
Walk operator is unitary
Number of non zero elements: 80


## Curved discriminant

The curved discriminant is defined as: $D = \square_{P^*}^\dagger S \square_P \\$ where each element is:

$D(x, y) = \sqrt{P^*(x, y)P(y, x)}$

Thus, we define it in the two ways and then do a sanity check for their equivalence.


In [16]:
# Defining the curved discriminant as the SPUE

D = square_rev.conj().T @ swap @ square # dim 4 x 4

# Sanity check for the equivalence

D_check = np.zeros((dim, dim))

for x in range(dim):
    for y in range(dim):

        D_check[x][y] = np.sqrt(P_rev[x, y]*P[y, x])

assert np.allclose(D, D_check), print("Critical error in the definition of the curved discriminant")

print("Curved discriminant correctly loaded")
print(D)



Curved discriminant correctly loaded
[[0.5        0.19014165 0.         0.25267796]
 [0.42073896 0.3        0.16045149 0.        ]
 [0.         0.31162066 0.6        0.24846637]
 [0.07915213 0.         0.36222206 0.5       ]]


## Spectral gap of the curved discriminant

A property of the curved discriminant is that it has both left and right singular vectors associated with singular value $1$ that exactly coincide with the sought probability distribution $\ket{\pi}. \\$
Before defining the chebyshev polynomial a crucial point is to find the spectral gap of the curved discriminant. Remember that the spectral gap is defined as $\delta = 1 - \sigma_2(D)$  where $\sigma_2(D)$ is the second largest singular value of $D$. 

In [21]:
# SVD of D

left_sv, svalues, right_sv = np.linalg.svd(D)

spectral_gap = 1 - svalues[1]

assert np.isclose(svalues[0], 1), print("Error: leading sv of D is not 1!")
print(f"Leading singular value of the curved discriminant is {svalues[0]}")
print(f"Second largest singular value: {svalues[1]}")
print(f"Spectral gap (The higher the better!): {spectral_gap}")

# Checking that both first right and left singular vector coincide with the pi state

left_vector = left_sv[:, 0]

right_vector = right_sv[0, :].conj()

if np.real(left_vector[0]) < 0:
    left_vector = -left_vector
if np.real(right_vector[0]) < 0:
    right_vector = -right_vector

assert np.allclose(left_vector, right_vector), "Error: right and left singular error are not equal"
print("Right and left singular vector are equal")

assert np.allclose(pi_state.flatten(), right_vector) and np.allclose(pi_state.flatten(), left_vector), "Error: right or left singular vector do not match pi state"
print("Right and left singular vector match pi state")



Leading singular value of the curved discriminant is 0.9999999999999999
Second largest singular value: 0.6260861813731153
Spectral gap (The higher the better!): 0.37391381862688466
Right and left singular vector are equal
Right and left singular vector match pi state


## Chebyshev polynomial

We now define the polynomial that will be applied in GQSP to $W$